<a href="https://colab.research.google.com/github/liamarganingrahayu/Tugas/blob/main/Analisis_Sentimen_Twitter_Transformer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import os
import pandas as pd
import torch
from torch.utils.data import Dataset
from sklearn.preprocessing import LabelEncoder
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments

# 1. Konfigurasi Path dan Koneksi Drive (Jika dijalankan di Google Colab)
from google.colab import drive
drive.mount('/content/drive')

PATH_TRAIN = "/content/drive/MyDrive/Colab Notebooks/Dataset /archive (2)/twitter_training.csv"
PATH_TEST = "/content/drive/MyDrive/Colab Notebooks/Dataset /archive (2)/twitter_validation.csv"

def prepare_data():
    # Membaca data pelatihan (Twitter Training CSV biasanya tidak memiliki header)
    # Kolom: [ID, Entity, Sentiment, Content]
    cols = ['id', 'entity', 'sentiment', 'text']
    df_train_full = pd.read_csv(PATH_TRAIN, names=cols).dropna(subset=['text'])

    # Sampling: 100 Positif, 100 Negatif, 100 Netral
    df_pos = df_train_full[df_train_full['sentiment'] == 'Positive'].sample(100, random_state=42)
    df_neg = df_train_full[df_train_full['sentiment'] == 'Negative'].sample(100, random_state=42)
    df_neu = df_train_full[df_train_full['sentiment'] == 'Neutral'].sample(100, random_state=42)

    train_df = pd.concat([df_pos, df_neg, df_neu]).reset_index(drop=True)

    # Membaca data pengujian (400 kalimat tanpa label/diabaikan labelnya)
    df_test_full = pd.read_csv(PATH_TEST, names=cols).dropna(subset=['text'])
    test_df = df_test_full.sample(400, random_state=42).reset_index(drop=True)

    return train_df, test_df

# 2. Dataset Custom untuk PyTorch
class TwitterDataset(Dataset):
    def __init__(self, encodings, labels=None):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        if self.labels is not None:
            item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.encodings['input_ids'])

def main():
    print("--- Memuat dan Menyiapkan Data ---")
    train_df, test_df = prepare_data()

    # Encode label string ke angka (Positive: 0, Negative: 1, Neutral: 2 dsb)
    le = LabelEncoder()
    train_df['label_idx'] = le.fit_transform(train_df['sentiment'])
    num_labels = len(le.classes_)

    # Inisialisasi Tokenizer BERT
    model_name = "bert-base-uncased"
    tokenizer = BertTokenizer.from_pretrained(model_name)

    print(f"--- Tokenisasi Kalimat (Model: {model_name}) ---")
    train_encodings = tokenizer(list(train_df['text']), truncation=True, padding=True, max_length=128)
    test_encodings = tokenizer(list(test_df['text']), truncation=True, padding=True, max_length=128)

    train_dataset = TwitterDataset(train_encodings, list(train_df['label_idx']))
    test_dataset = TwitterDataset(test_encodings)

    # 3. Konfigurasi Model Transformer
    model = BertForSequenceClassification.from_pretrained(model_name, num_labels=num_labels)

    training_args = TrainingArguments(
        output_dir='./results',
        num_train_epochs=5,              # Melakukan iterasi lebih banyak karena data sedikit
        per_device_train_batch_size=16,
        logging_dir='./logs',
        logging_steps=10,
        report_to="none"
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset
    )

    print("--- Memulai Pelatihan Model (Fine-tuning) ---")
    trainer.train()

    # 4. Melakukan Prediksi (Pelabelan) pada Data Pengujian
    print("--- Melakukan Pelabelan pada 400 Kalimat Pengujian ---")
    raw_predictions = trainer.predict(test_dataset)

    # Mengambil indeks dengan probabilitas tertinggi
    predicted_labels_idx = raw_predictions.predictions.argmax(axis=1)

    # Mengembalikan angka ke label string asli
    predicted_sentiments = le.inverse_transform(predicted_labels_idx)

    # Menampilkan Hasil
    test_df['predicted_sentiment'] = predicted_sentiments

    print("\n--- Contoh Hasil Pelabelan ---")
    print(test_df[['text', 'predicted_sentiment']].head(10))

    # Simpan hasil ke CSV
    output_path = "hasil_pelabelan_transformer.csv"
    test_df.to_csv(output_path, index=False)
    print(f"\nHasil lengkap telah disimpan ke: {output_path}")

if __name__ == "__main__":
    main()

Mounted at /content/drive
--- Memuat dan Menyiapkan Data ---


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

...

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss
10,1.116300
20,1.106000
30,0.995400
40,0.869700
50,0.649700
60,0.502100
70,0.417400
80,0.273900
90,0.236800


--- Melakukan Pelabelan pada 400 Kalimat Pengujian ---


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)



--- Contoh Hasil Pelabelan ---
                                                text predicted_sentiment
0  Remote working and an increase in cloud-based ...            Positive
1  I actually quite like the design of the ps5. I...            Positive
2  New York charges Johnson & Johnson with insura...             Neutral
3         Chris loves me in borderlands one and two.            Positive
4  Check out my video! #LeagueofLegends | Capture...             Neutral
5  Amazing deal for you!\n\nLenovo Legion Y540 9t...             Neutral
6  [PS4] | Assassins Creed Syndicate First Playth...             Neutral
7                         @EAMaddenNFL servers down?            Negative
8          @SpeakerPelosi this is VERY INTERESTING 🧐            Positive
9  So good I had to share! Check out all the item...            Positive

Hasil lengkap telah disimpan ke: hasil_pelabelan_transformer.csv
